In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, 
                            accuracy_score, balanced_accuracy_score,
                            precision_recall_fscore_support)
import time
import copy
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# TensorBoard
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

# ============================================================
# 1. CONFIGURATION ET CHARGEMENT DES MÉTADONNÉES
# ============================================================

print("\n" + "="*70)
print("🚀 ENTRAÎNEMENT - CNN, RESNET, VIT")
print("   Utilisation des données prétraitées")
print("="*70)

# CORRECTION DES CHEMINS - Utilisation du bon nom d'utilisateur
BASE_DIR = "MultimodalAI/Alzheimer/data"
PRETRAITE_DIR = os.path.join(BASE_DIR, "pretraite")
TRAIN_DIR = os.path.join(PRETRAITE_DIR, "train")
CONFIG_FILE = os.path.join(PRETRAITE_DIR, "dataset_config.json")

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️  Configuration:")
print(f"   Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()
    print(f"   ✅ Cache GPU vidé")

# Charger la configuration
print(f"\n📋 Chargement de la configuration...")
if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        saved_config = json.load(f)
    
    print(f"   ✅ Configuration chargée")
    print(f"   Classes: {saved_config['dataset_info']['classes']}")
    print(f"   Total images: {saved_config['dataset_info']['total_train_preprocessed']}")
    print(f"   Déséquilibre: {saved_config['dataset_info']['imbalance_ratio']:.1f}:1")
    
    # Class weights
    class_weights_dict = saved_config['class_weights']
    print(f"\n   ⚖️ Class weights:")
    for cls, weight in class_weights_dict.items():
        print(f"      {cls}: {weight:.2f}")
else:
    print(f"   ⚠️ Fichier de configuration introuvable")
    saved_config = None

# Configuration d'entraînement
CONFIG = {
    'train_dir': TRAIN_DIR,
    'img_size': 224,
    'batch_size': 32,
    'num_workers': 4,
    'num_epochs': 25,
    'val_split': 0.2,
    'learning_rate_cnn': 0.001,
    'learning_rate_resnet': 0.0001,
    'learning_rate_vit': 0.00005,
}

# CORRECTION TENSORBOARD - Chemin absolu explicite
TENSORBOARD_DIR = os.path.join(BASE_DIR, "runs")
os.makedirs(TENSORBOARD_DIR, exist_ok=True)

# TEST TENSORBOARD - Vérification avant entraînement
print(f"\n📊 Configuration TensorBoard:")
print(f"   Log directory: {TENSORBOARD_DIR}")
print(f"   Le répertoire existe: {os.path.exists(TENSORBOARD_DIR)}")

# Test de création de fichier TensorBoard
test_writer = SummaryWriter(os.path.join(TENSORBOARD_DIR, "test_initial"))
test_writer.add_scalar('Test/Initial', 1.0, 1)
test_writer.flush()
test_writer.close()

# Vérifier les fichiers créés
import glob
log_files = glob.glob(os.path.join(TENSORBOARD_DIR, "**", "*.tfevents.*"), recursive=True)
print(f"   Fichiers de log trouvés: {len(log_files)}")
for file in log_files:
    print(f"      📄 {os.path.basename(file)}")

print(f"   Pour visualiser: tensorboard --logdir={TENSORBOARD_DIR}")

print(f"\n⚙️ Hyperparamètres:")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Époques: {CONFIG['num_epochs']}")
print(f"   Val split: {CONFIG['val_split']*100:.0f}%")

# ============================================================
# 2. DATASET PERSONNALISÉ
# ============================================================

class AlzheimerDataset(Dataset):
    """Dataset pour images prétraitées"""
    def __init__(self, data_dir, transform=None, indices=None):
        self.data_dir = data_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        self.idx_to_class = {}
        
        # Charger les classes
        classes = [d for d in os.listdir(data_dir) 
                  if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('_')]
        classes.sort()
        
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        self.idx_to_class = {idx: cls for idx, cls in enumerate(classes)}
        
        # Charger toutes les images
        for class_name in classes:
            class_dir = os.path.join(data_dir, class_name)
            images = [f for f in os.listdir(class_dir) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            
            for img_name in images:
                self.images.append(os.path.join(class_dir, img_name))
                self.labels.append(self.class_to_idx[class_name])
        
        # Filtrer par indices si fourni (pour train/val split)
        if indices is not None:
            self.images = [self.images[i] for i in indices]
            self.labels = [self.labels[i] for i in indices]
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# ============================================================
# 3. TRANSFORMATIONS (AUGMENTATION POUR ENTRAÎNEMENT)
# ============================================================

print("\n" + "="*70)
print("🎨 DÉFINITION DES TRANSFORMATIONS")
print("="*70)

# Les images sont déjà redimensionnées et normalisées
# On applique juste l'augmentation pour le train
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    # Re-normalisation car images sauvegardées sont dénormalisées
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Transformations définies")
print("   Train: Augmentation + Normalize")
print("   Val: Normalize seulement")

# ============================================================
# 4. CRÉATION DU SPLIT TRAIN/VAL
# ============================================================

print("\n" + "="*70)
print("✂️ CRÉATION DU SPLIT TRAIN/VALIDATION")
print("="*70)

# Charger toutes les données pour faire le split
temp_dataset = AlzheimerDataset(TRAIN_DIR)
targets = np.array(temp_dataset.labels)

print(f"\n📊 Dataset complet:")
print(f"   Total: {len(temp_dataset)} images")
print(f"   Classes: {list(temp_dataset.class_to_idx.keys())}")

# Distribution
unique, counts = np.unique(targets, return_counts=True)
print(f"\n   Distribution:")
for cls, count in zip(unique, counts):
    class_name = temp_dataset.idx_to_class[cls]
    print(f"      {class_name}: {count} ({count/len(targets)*100:.1f}%)")

# Split stratifié
train_indices, val_indices = train_test_split(
    range(len(temp_dataset)),
    test_size=CONFIG['val_split'],
    stratify=targets,
    random_state=42
)

print(f"\n✅ Split stratifié:")
print(f"   Train: {len(train_indices)} images ({(1-CONFIG['val_split'])*100:.0f}%)")
print(f"   Val: {len(val_indices)} images ({CONFIG['val_split']*100:.0f}%)")

# Créer les datasets
train_dataset = AlzheimerDataset(TRAIN_DIR, transform=train_transform, indices=train_indices)
val_dataset = AlzheimerDataset(TRAIN_DIR, transform=val_transform, indices=val_indices)

# ============================================================
# 5. CLASS WEIGHTS ET WEIGHTED SAMPLER
# ============================================================

print("\n" + "="*70)
print("⚖️ CALCUL DES CLASS WEIGHTS ET SAMPLER")
print("="*70)

# Calculer les class weights sur le train set
train_targets = targets[train_indices]
class_counts = np.bincount(train_targets)

# Effective Number of Samples (pour déséquilibre extrême)
beta = 0.9999
effective_num = 1.0 - np.power(beta, class_counts)
class_weights = (1.0 - beta) / effective_num
class_weights = class_weights / class_weights.sum() * len(class_counts)

print("📊 Class weights (Effective Number):")
for cls_idx, weight in enumerate(class_weights):
    print(f"   {train_dataset.idx_to_class[cls_idx]}: {weight:.2f}")

class_weights_tensor = torch.FloatTensor(class_weights)

# Weighted Random Sampler
sample_weights = [class_weights[target] for target in train_targets]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("\n✅ WeightedRandomSampler créé")

# ============================================================
# 6. DATALOADERS
# ============================================================

print("\n" + "="*70)
print("📦 CRÉATION DES DATALOADERS")
print("="*70)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    sampler=sampler,
    num_workers=CONFIG['num_workers'],
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"✅ Dataloaders créés:")
print(f"   Train: {len(train_loader)} batches")
print(f"   Val: {len(val_loader)} batches")

# Test d'un batch
test_imgs, test_labels = next(iter(train_loader))
print(f"\n🔍 Test batch:")
print(f"   Shape: {test_imgs.shape}")
print(f"   Range: [{test_imgs.min():.2f}, {test_imgs.max():.2f}]")

del test_imgs, test_labels
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ============================================================
# 7. FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):
    """Focal Loss pour déséquilibre extrême"""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(
            inputs, targets, reduction='none', weight=self.alpha
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss)
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

print("\n✅ Focal Loss définie")

# ============================================================
# 8. ARCHITECTURES DES MODÈLES
# ============================================================

print("\n" + "="*70)
print("🏗️ DÉFINITION DES ARCHITECTURES")
print("="*70)

num_classes = len(train_dataset.class_to_idx)

# ========== CNN PERSONNALISÉ ==========
class CustomCNN(nn.Module):
    """CNN optimisé pour images médicales"""
    def __init__(self, num_classes=4):
        super(CustomCNN, self).__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1),
            
            # Block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),
            
            # Block 3
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3),
            
            # Block 4
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.4),
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print("✅ CustomCNN défini")

# ========== RESNET-101 ==========
def get_resnet101(num_classes, pretrained=True):
    """ResNet-101 avec transfer learning"""
    model = models.resnet101(pretrained=pretrained)
    
    # Geler les premières couches
    for param in list(model.parameters())[:-30]:
        param.requires_grad = False
    
    # Remplacer le classifier
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.6),
        nn.Linear(num_ftrs, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes)
    )
    
    return model

print("✅ ResNet-101 défini")

# ========== VISION TRANSFORMER ==========
def get_vit(num_classes, pretrained=True):
    """Vision Transformer (ViT-B/16)"""
    try:
        model = models.vit_b_16(pretrained=pretrained)
        
        # Remplacer la tête de classification
        num_ftrs = model.heads.head.in_features
        model.heads.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(num_ftrs, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        
        return model
    except Exception as e:
        print(f"⚠️ Erreur ViT: {e}")
        return None

print("✅ ViT défini")

# ============================================================
# 9. FONCTION D'ENTRAÎNEMENT AVEC TENSORBOARD CORRIGÉE
# ============================================================

def train_model(model, train_loader, val_loader, criterion, optimizer, 
                scheduler, num_epochs, device, model_name):
    """Entraîne un modèle avec métriques détaillées et TensorBoard"""
    
    # CORRECTION : Définition correcte de log_dir
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir = os.path.join(TENSORBOARD_DIR, f'{model_name}_{timestamp}')
    
    # Créer le writer TensorBoard
    writer = SummaryWriter(log_dir)
    
    # DIAGNOSTIC TensorBoard
    print(f"📊 TensorBoard log directory: {log_dir}")
    print(f"   Le répertoire existe: {os.path.exists(log_dir)}")
    
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_balanced_acc = 0.0
    
    history = {
        'train_loss': [], 'train_acc': [], 'train_balanced_acc': [],
        'val_loss': [], 'val_acc': [], 'val_balanced_acc': [],
        'learning_rates': []
    }
    
    print(f"\n{'='*70}")
    print(f"🚀 ENTRAÎNEMENT: {model_name}")
    print(f"{'='*70}")
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        print(f'\nÉpoque {epoch+1}/{num_epochs}')
        print('-' * 60)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader
            
            running_loss = 0.0
            all_preds = []
            all_labels = []
            
            # Barre de progression
            batch_count = 0
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
                batch_count += 1
                if batch_count % 50 == 0:
                    print(f'   {phase} [{batch_count}/{len(dataloader)}] Loss: {loss.item():.4f}', end='\r')
            
            # Métriques
            epoch_loss = running_loss / len(dataloader.dataset)
            acc = accuracy_score(all_labels, all_preds)
            balanced_acc = balanced_accuracy_score(all_labels, all_preds)
            
            # Sauvegarder l'historique
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(acc)
                history['train_balanced_acc'].append(balanced_acc)
                history['learning_rates'].append(optimizer.param_groups[0]['lr'])
                
                # TensorBoard - Train avec flush
                writer.add_scalar('Loss/train', epoch_loss, epoch)
                writer.add_scalar('Accuracy/train', acc, epoch)
                writer.add_scalar('Balanced_Accuracy/train', balanced_acc, epoch)
                writer.add_scalar('Learning_Rate', optimizer.param_groups[0]['lr'], epoch)
                writer.flush()  # FORCE L'ÉCRITURE
                
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(acc)
                history['val_balanced_acc'].append(balanced_acc)
                
                # TensorBoard - Validation avec flush
                writer.add_scalar('Loss/val', epoch_loss, epoch)
                writer.add_scalar('Accuracy/val', acc, epoch)
                writer.add_scalar('Balanced_Accuracy/val', balanced_acc, epoch)
                writer.flush()  # FORCE L'ÉCRITURE
            
            print(f'{phase:5s} | Loss: {epoch_loss:.4f} | Acc: {acc:.4f} | Bal.Acc: {balanced_acc:.4f}')
            
            # Sauvegarder le meilleur modèle
            if phase == 'val' and balanced_acc > best_balanced_acc:
                best_balanced_acc = balanced_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                print(f'      ✅ Nouveau meilleur! Bal.Acc: {best_balanced_acc:.4f}')
                
                # TensorBoard - Meilleur modèle
                writer.add_scalar('Best_Balanced_Accuracy', best_balanced_acc, epoch)
                writer.flush()
        
        # TensorBoard - Comparaison Train vs Val
        writer.add_scalars('Loss/train_vs_val', {
            'train': history['train_loss'][-1],
            'val': history['val_loss'][-1]
        }, epoch)
        
        writer.add_scalars('Accuracy/train_vs_val', {
            'train': history['train_acc'][-1],
            'val': history['val_acc'][-1]
        }, epoch)
        writer.flush()
        
        # Scheduler
        if scheduler:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(history['val_balanced_acc'][-1])
            else:
                scheduler.step()
        
        epoch_time = time.time() - epoch_start
        print(f'Temps: {epoch_time//60:.0f}m {epoch_time%60:.0f}s')
        
        # TensorBoard - Temps par époque
        writer.add_scalar('Time/epoch_duration', epoch_time, epoch)
        writer.flush()
        
        # Sauvegarder checkpoint
        if (epoch + 1) % 5 == 0:
            checkpoint_name = f'checkpoint_{model_name.lower().replace(" ", "_")}_epoch{epoch+1}.pth'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': epoch_loss,
                'balanced_acc': balanced_acc,
                'history': history
            }, checkpoint_name)
            print(f'💾 Checkpoint: {checkpoint_name}')
    
    total_time = time.time() - since
    print(f'\n{"="*70}')
    print(f'✅ Entraînement terminé en {total_time//60:.0f}m {total_time%60:.0f}s')
    print(f'🏆 Meilleure Balanced Accuracy: {best_balanced_acc:.4f}')
    print(f'{"="*70}')
    
    # TensorBoard - Temps total
    writer.add_text('Training/total_time', f'{total_time//60:.0f}m {total_time%60:.0f}s')
    writer.add_text('Training/best_balanced_acc', f'{best_balanced_acc:.4f}')
    writer.flush()
    
    # Fermer le writer
    writer.close()
    
    # Vérification des fichiers TensorBoard créés
    model_log_files = glob.glob(os.path.join(log_dir, "*.tfevents.*"))
    print(f"📁 Fichiers TensorBoard créés pour {model_name}: {len(model_log_files)}")
    
    # Charger les meilleurs poids
    model.load_state_dict(best_model_wts)
    
    return model, history

# ============================================================
# 10. FONCTION D'ÉVALUATION
# ============================================================

def evaluate_model(model, dataloader, device, class_names):
    """Évaluation complète avec toutes les métriques"""
    
    print("\n📊 Évaluation détaillée...")
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Métriques
    acc = accuracy_score(all_labels, all_preds)
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(
        all_labels, all_preds, target_names=class_names, zero_division=0
    )
    
    print(f"\n📈 Résultats:")
    print(f"   Accuracy: {acc:.4f}")
    print(f"   Balanced Accuracy: {balanced_acc:.4f} ⭐")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall: {recall:.4f}")
    print(f"   F1-Score: {f1:.4f}")
    
    print(f"\n{report}")
    
    return {
        'accuracy': acc,
        'balanced_accuracy': balanced_acc,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm,
        'classification_report': report,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs
    }

# ============================================================
# 11. ENTRAÎNEMENT DES 3 MODÈLES
# ============================================================

print("\n" + "="*70)
print("🎯 ENTRAÎNEMENT DES 3 MODÈLES")
print("="*70)

class_names = list(train_dataset.class_to_idx.keys())
models_results = {}
histories = {}

# ========== 1. CNN ==========
print("\n" + "🔵"*35)
print("MODÈLE 1/3: Custom CNN")
print("🔵"*35)

try:
    model_cnn = CustomCNN(num_classes=num_classes).to(device)
    criterion_cnn = FocalLoss(alpha=class_weights_tensor.to(device), gamma=2.0)
    optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=CONFIG['learning_rate_cnn'])
    scheduler_cnn = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_cnn, mode='max', factor=0.5, patience=3, verbose=True
    )
    
    model_cnn, history_cnn = train_model(
        model_cnn, train_loader, val_loader, criterion_cnn, optimizer_cnn,
        scheduler_cnn, CONFIG['num_epochs'], device, 'CNN'
    )
    
    # Sauvegarder
    torch.save(model_cnn.state_dict(), 'best_cnn.pth')
    print("\n💾 Modèle sauvegardé: best_cnn.pth")
    
    # Évaluer
    results_cnn = evaluate_model(model_cnn, val_loader, device, class_names)
    
    models_results['CNN'] = results_cnn
    histories['CNN'] = history_cnn
    
    # Libérer mémoire
    del model_cnn
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
except Exception as e:
    print(f"❌ Erreur CNN: {e}")
    import traceback
    traceback.print_exc()

# ========== 2. RESNET-101 ==========
print("\n" + "🟢"*35)
print("MODÈLE 2/3: ResNet-101")
print("🟢"*35)

try:
    model_resnet = get_resnet101(num_classes=num_classes, pretrained=True).to(device)
    criterion_resnet = FocalLoss(alpha=class_weights_tensor.to(device), gamma=2.0)
    optimizer_resnet = optim.Adam(model_resnet.parameters(), lr=CONFIG['learning_rate_resnet'])
    scheduler_resnet = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_resnet, mode='max', factor=0.5, patience=3, verbose=True
    )
    
    model_resnet, history_resnet = train_model(
        model_resnet, train_loader, val_loader, criterion_resnet, optimizer_resnet,
        scheduler_resnet, CONFIG['num_epochs'], device, 'ResNet-101'
    )
    
    # Sauvegarder
    torch.save(model_resnet.state_dict(), 'best_resnet101.pth')
    print("\n💾 Modèle sauvegardé: best_resnet101.pth")
    
    # Évaluer
    results_resnet = evaluate_model(model_resnet, val_loader, device, class_names)
    
    models_results['ResNet-101'] = results_resnet
    histories['ResNet-101'] = history_resnet
    
    # Libérer mémoire
    del model_resnet
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
except Exception as e:
    print(f"❌ Erreur ResNet: {e}")
    import traceback
    traceback.print_exc()

# ========== 3. VIT ==========
print("\n" + "🟣"*35)
print("MODÈLE 3/3: Vision Transformer")
print("🟣"*35)

try:
    model_vit = get_vit(num_classes=num_classes, pretrained=True)
    
    if model_vit is not None:
        model_vit = model_vit.to(device)
        criterion_vit = FocalLoss(alpha=class_weights_tensor.to(device), gamma=2.0)
        optimizer_vit = optim.AdamW(
            model_vit.parameters(), 
            lr=CONFIG['learning_rate_vit'], 
            weight_decay=0.01
        )
        scheduler_vit = optim.lr_scheduler.CosineAnnealingLR(
            optimizer_vit, T_max=CONFIG['num_epochs']
        )
        
        model_vit, history_vit = train_model(
            model_vit, train_loader, val_loader, criterion_vit, optimizer_vit,
            scheduler_vit, CONFIG['num_epochs'], device, 'ViT'
        )
        
        # Sauvegarder
        torch.save(model_vit.state_dict(), 'best_vit.pth')
        print("\n💾 Modèle sauvegardé: best_vit.pth")
        
        # Évaluer
        results_vit = evaluate_model(model_vit, val_loader, device, class_names)
        
        models_results['ViT'] = results_vit
        histories['ViT'] = history_vit
        
        # Libérer mémoire
        del model_vit
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("⚠️ ViT non disponible")
    
except Exception as e:
    print(f"❌ Erreur ViT: {e}")
    import traceback
    traceback.print_exc()

# ============================================================
# 12. COMPARAISONS ET VISUALISATIONS
# ============================================================

print("\n" + "="*70)
print("📊 COMPARAISONS DES MODÈLES")
print("="*70)

if len(models_results) == 0:
    print("❌ Aucun modèle entraîné avec succès")
else:
    # ========== Tableau comparatif ==========
    print("\n📋 TABLEAU COMPARATIF:")
    print(f"\n{'Modèle':<15} {'Accuracy':<12} {'Bal.Acc':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
    print("-"*85)
    
    for model_name, results in models_results.items():
        print(f"{model_name:<15} "
              f"{results['accuracy']:<12.4f} "
              f"{results['balanced_accuracy']:<12.4f} "
              f"{results['precision']:<12.4f} "
              f"{results['recall']:<12.4f} "
              f"{results['f1_score']:<12.4f}")
    
    print("="*85)
    
    # Meilleur modèle
    best_model = max(models_results.items(), key=lambda x: x[1]['balanced_accuracy'])
    print(f"\n🏆 MEILLEUR MODÈLE: {best_model[0]}")
    print(f"   Balanced Accuracy: {best_model[1]['balanced_accuracy']:.4f}")
    print(f"   F1-Score: {best_model[1]['f1_score']:.4f}")
    
    # ========== Graphiques de comparaison ==========
    print("\n📊 Génération des graphiques...")
    
    # 1. Comparaison des Loss
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Train Loss
    ax = axes[0, 0]
    for model_name, history in histories.items():
        ax.plot(history['train_loss'], label=model_name, linewidth=2, marker='o', markersize=4)
    ax.set_title('Loss d\'Entraînement', fontsize=14, fontweight='bold')
    ax.set_xlabel('Époque')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Val Loss
    ax = axes[0, 1]
    for model_name, history in histories.items():
        ax.plot(history['val_loss'], label=model_name, linewidth=2, marker='o', markersize=4)
    ax.set_title('Loss de Validation', fontsize=14, fontweight='bold')
    ax.set_xlabel('Époque')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Train Balanced Acc
    ax = axes[1, 0]
    for model_name, history in histories.items():
        ax.plot(history['train_balanced_acc'], label=model_name, linewidth=2, marker='o', markersize=4)
    ax.set_title('Balanced Accuracy d\'Entraînement', fontsize=14, fontweight='bold')
    ax.set_xlabel('Époque')
    ax.set_ylabel('Balanced Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Val Balanced Acc
    ax = axes[1, 1]
    for model_name, history in histories.items():
        ax.plot(history['val_balanced_acc'], label=model_name, linewidth=2, marker='o', markersize=4)
    ax.set_title('Balanced Accuracy de Validation', fontsize=14, fontweight='bold')
    ax.set_xlabel('Époque')
    ax.set_ylabel('Balanced Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_comparison.png', dpi=300, bbox_inches='tight')
    print("✅ Graphique sauvegardé: training_comparison.png")
    plt.show()
    
    # 2. Matrices de confusion
    n_models = len(models_results)
    fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 5))
    
    if n_models == 1:
        axes = [axes]
    
    for ax, (model_name, results) in zip(axes, models_results.items()):
        cm = results['confusion_matrix']
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_names, yticklabels=class_names, ax=ax,
                    cbar_kws={'label': 'Nombre de prédictions'})
        
        ax.set_title(f'{model_name}\nBal.Acc: {results["balanced_accuracy"]:.4f}', 
                     fontsize=12, fontweight='bold')
        ax.set_ylabel('Vraie classe')
        ax.set_xlabel('Classe prédite')
    
    plt.tight_layout()
    plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
    print("✅ Graphique sauvegardé: confusion_matrices.png")
    plt.show()
    
    # 3. Graphique des métriques finales
    fig, ax = plt.subplots(figsize=(12, 6))
    
    metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score']
    metric_labels = ['Accuracy', 'Bal. Accuracy', 'Precision', 'Recall', 'F1-Score']
    
    x = np.arange(len(metric_labels))
    width = 0.25
    
    for i, (model_name, results) in enumerate(models_results.items()):
        values = [results[metric] for metric in metrics]
        ax.bar(x + i*width, values, width, label=model_name)
    
    ax.set_xlabel('Métriques', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Comparaison des Métriques par Modèle', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(metric_labels)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.0])
    
    plt.tight_layout()
    plt.savefig('metrics_comparison.png', dpi=300, bbox_inches='tight')
    print("✅ Graphique sauvegardé: metrics_comparison.png")
    plt.show()
    
    # ========== Sauvegarder les résultats ==========
    print("\n💾 Sauvegarde des résultats...")
    
    results_summary = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'config': CONFIG,
        'models': {}
    }
    
    for model_name, results in models_results.items():
        results_summary['models'][model_name] = {
            'accuracy': float(results['accuracy']),
            'balanced_accuracy': float(results['balanced_accuracy']),
            'precision': float(results['precision']),
            'recall': float(results['recall']),
            'f1_score': float(results['f1_score']),
            'confusion_matrix': results['confusion_matrix'].tolist(),
            'classification_report': results['classification_report']
        }
    
    results_file = os.path.join(PRETRAITE_DIR, 'training_results.json')
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(results_summary, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Résultats sauvegardés: {results_file}")

# ============================================================
# 13. RAPPORT FINAL ET INSTRUCTIONS TENSORBOARD
# ============================================================

print("\n" + "="*70)
print("🎉 ENTRAÎNEMENT TERMINÉ - RAPPORT FINAL")
print("="*70)

if len(models_results) > 0:
    print(f"\n📊 RÉSUMÉ:")
    print(f"   Modèles entraînés: {len(models_results)}")
    print(f"   Époques: {CONFIG['num_epochs']}")
    print(f"   Batch size: {CONFIG['batch_size']}")
    print(f"   Device: {device}")
    
    print(f"\n🏆 CLASSEMENT (Balanced Accuracy):")
    sorted_models = sorted(models_results.items(), 
                          key=lambda x: x[1]['balanced_accuracy'], 
                          reverse=True)
    for i, (model_name, results) in enumerate(sorted_models, 1):
        print(f"   {i}. {model_name}: {results['balanced_accuracy']:.4f}")
    
    print(f"\n💾 FICHIERS GÉNÉRÉS:")
    print(f"   - best_cnn.pth (si entraîné)")
    print(f"   - best_resnet101.pth (si entraîné)")
    print(f"   - best_vit.pth (si entraîné)")
    print(f"   - training_comparison.png")
    print(f"   - confusion_matrices.png")
    print(f"   - metrics_comparison.png")
    print(f"   - training_results.json")
    print(f"   - Checkpoints tous les 5 époques")
    print(f"   - Logs TensorBoard dans: {TENSORBOARD_DIR}")
    
    # VÉRIFICATION FINALE TENSORBOARD
    print(f"\n📊 VÉRIFICATION TENSORBOARD FINALE:")
    all_log_files = glob.glob(os.path.join(TENSORBOARD_DIR, "**", "*.tfevents.*"), recursive=True)
    print(f"   Total fichiers de log: {len(all_log_files)}")
    
    if len(all_log_files) > 0:
        print(f"   ✅ TensorBoard est configuré correctement!")
        print(f"\n🎯 INSTRUCTIONS TENSORBOARD:")
        print(f"   1. Ouvrez un terminal/CMD")
        print(f"   2. Exécutez: tensorboard --logdir={TENSORBOARD_DIR}")
        print(f"   3. Ouvrez votre navigateur sur: http://localhost:6006")
        print(f"   4. Les données devraient apparaître dans les onglets:")
        print(f"      - SCALARS: Loss, Accuracy, Learning Rate")
        print(f"      - GRAPHS: Architecture des modèles")
        print(f"      - HISTOGRAMS: Distributions des poids")
    else:
        print(f"   ⚠️ Aucun fichier de log trouvé. Vérifiez les permissions d'écriture.")
    
    print(f"\n💡 UTILISATION DES MODÈLES:")
    print(f"   # Charger un modèle")
    print(f"   model = CustomCNN(num_classes={num_classes})")
    print(f"   model.load_state_dict(torch.load('best_cnn.pth'))")
    print(f"   model.eval()")
    
    print(f"\n🔍 ANALYSE DES RÉSULTATS:")
    best = sorted_models[0]
    print(f"   Le meilleur modèle est {best[0]} avec:")
    print(f"   - Balanced Accuracy: {best[1]['balanced_accuracy']:.4f}")
    print(f"   - F1-Score: {best[1]['f1_score']:.4f}")
    
    if best[1]['balanced_accuracy'] < 0.7:
        print(f"\n   ⚠️ Performances modérées. Suggestions:")
        print(f"      - Augmenter le nombre d'époques")
        print(f"      - Ajuster les hyperparamètres")
        print(f"      - Essayer d'autres architectures")
    elif best[1]['balanced_accuracy'] < 0.85:
        print(f"\n   ✅ Bonnes performances. Améliorations possibles:")
        print(f"      - Fine-tuning des hyperparamètres")
        print(f"      - Augmentation de données plus agressive")
    else:
        print(f"\n   🎉 Excellentes performances!")

print("\n" + "="*70)


🚀 ENTRAÎNEMENT - CNN, RESNET, VIT
   Utilisation des données prétraitées

🖥️  Configuration:
   Device: cpu

📋 Chargement de la configuration...
   ⚠️ Fichier de configuration introuvable

📊 Configuration TensorBoard:
   Log directory: C:/Users/angej/Desktop/MultimodalAI/Alzheimer/data\runs
   Le répertoire existe: True
   Fichiers de log trouvés: 2
      📄 events.out.tfevents.1763423940.ANGE-JULES.30928.0
      📄 events.out.tfevents.1763465872.ANGE-JULES.22000.0
   Pour visualiser: tensorboard --logdir=C:/Users/angej/Desktop/MultimodalAI/Alzheimer/data\runs

⚙️ Hyperparamètres:
   Batch size: 32
   Époques: 25
   Val split: 20%

🎨 DÉFINITION DES TRANSFORMATIONS
✅ Transformations définies
   Train: Augmentation + Normalize
   Val: Normalize seulement

✂️ CRÉATION DU SPLIT TRAIN/VALIDATION

📊 Dataset complet:
   Total: 11521 images
   Classes: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

   Distribution:
      MildDemented: 1665 (14.5%)
      ModerateDemente

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, 
                            accuracy_score, balanced_accuracy_score,
                            precision_recall_fscore_support)
import time
import copy
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CONFIGURATION ET CHARGEMENT DES MÉTADONNÉES
# ============================================================

print("\n" + "="*70)
print("🚀 ENTRAÎNEMENT - CNN, RESNET, VIT")
print("   Utilisation des données prétraitées")
print("="*70)

# Chemins
BASE_DIR = "MultimodalAI/Alzheimer/data"
PRETRAITE_DIR = os.path.join(BASE_DIR, "pretraite")
TRAIN_DIR = os.path.join(PRETRAITE_DIR, "train")
CONFIG_FILE = os.path.join(PRETRAITE_DIR, "dataset_config.json")

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️  Configuration:")
print(f"   Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()
    print(f"   ✅ Cache GPU vidé")

# Charger la configuration
print(f"\n📋 Chargement de la configuration...")
if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        saved_config = json.load(f)
    
    print(f"   ✅ Configuration chargée")
    print(f"   Classes: {saved_config['dataset_info']['classes']}")
    print(f"   Total images: {saved_config['dataset_info']['total_train_preprocessed']}")
    print(f"   Déséquilibre: {saved_config['dataset_info']['imbalance_ratio']:.1f}:1")
    
    # Class weights
    class_weights_dict = saved_config['class_weights']
    print(f"\n   ⚖️ Class weights:")
    for cls, weight in class_weights_dict.items():
        print(f"      {cls}: {weight:.2f}")
else:
    print(f"   ⚠️ Fichier de configuration introuvable")
    saved_config = None

# Configuration d'entraînement
CONFIG = {
    'train_dir': TRAIN_DIR,
    'img_size': 224,
    'batch_size': 32,
    'num_workers': 4,
    'num_epochs': 25,
    'val_split': 0.2,
    'learning_rate_cnn': 0.001,
    'learning_rate_resnet': 0.0001,
    'learning_rate_vit': 0.00005,
}

print(f"\n⚙️ Hyperparamètres:")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Époques: {CONFIG['num_epochs']}")
print(f"   Val split: {CONFIG['val_split']*100:.0f}%")

# ============================================================
# 2. DATASET PERSONNALISÉ
# ============================================================

class AlzheimerDataset(Dataset):
    """Dataset pour images prétraitées"""
    def __init__(self, data_dir, transform=None, indices=None):
        self.data_dir = data_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        self.idx_to_class = {}
        
        # Charger les classes
        classes = [d for d in os.listdir(data_dir) 
                  if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('_')]
        classes.sort()
        
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        self.idx_to_class = {idx: cls for idx, cls in enumerate(classes)}
        
        # Charger toutes les images
        for class_name in classes:
            class_dir = os.path.join(data_dir, class_name)
            images = [f for f in os.listdir(class_dir) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            
            for img_name in images:
                self.images.append(os.path.join(class_dir, img_name))
                self.labels.append(self.class_to_idx[class_name])
        
        # Filtrer par indices si fourni (pour train/val split)
        if indices is not None:
            self.images = [self.images[i] for i in indices]
            self.labels = [self.labels[i] for i in indices]
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# ============================================================
# 3. TRANSFORMATIONS (AUGMENTATION POUR ENTRAÎNEMENT)
# ============================================================

print("\n" + "="*70)
print("🎨 DÉFINITION DES TRANSFORMATIONS")
print("="*70)

# Les images sont déjà redimensionnées et normalisées
# On applique juste l'augmentation pour le train
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    # Re-normalisation car images sauvegardées sont dénormalisées
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Transformations définies")
print("   Train: Augmentation + Normalize")
print("   Val: Normalize seulement")

# ============================================================
# 4. CRÉATION DU SPLIT TRAIN/VAL
# ============================================================

print("\n" + "="*70)
print("✂️ CRÉATION DU SPLIT TRAIN/VALIDATION")
print("="*70)

# Charger toutes les données pour faire le split
temp_dataset = AlzheimerDataset(TRAIN_DIR)
targets = np.array(temp_dataset.labels)

print(f"\n📊 Dataset complet:")
print(f"   Total: {len(temp_dataset)} images")
print(f"   Classes: {list(temp_dataset.class_to_idx.keys())}")

# Distribution
unique, counts = np.unique(targets, return_counts=True)
print(f"\n   Distribution:")
for cls, count in zip(unique, counts):
    class_name = temp_dataset.idx_to_class[cls]
    print(f"      {class_name}: {count} ({count/len(targets)*100:.1f}%)")

# Split stratifié
train_indices, val_indices = train_test_split(
    range(len(temp_dataset)),
    test_size=CONFIG['val_split'],
    stratify=targets,
    random_state=42
)

print(f"\n✅ Split stratifié:")
print(f"   Train: {len(train_indices)} images ({(1-CONFIG['val_split'])*100:.0f}%)")
print(f"   Val: {len(val_indices)} images ({CONFIG['val_split']*100:.0f}%)")

# Créer les datasets
train_dataset = AlzheimerDataset(TRAIN_DIR, transform=train_transform, indices=train_indices)
val_dataset = AlzheimerDataset(TRAIN_DIR, transform=val_transform, indices=val_indices)

# ============================================================
# 5. CLASS WEIGHTS ET WEIGHTED SAMPLER
# ============================================================

print("\n" + "="*70)
print("⚖️ CALCUL DES CLASS WEIGHTS ET SAMPLER")
print("="*70)

# Calculer les class weights sur le train set
train_targets = targets[train_indices]
class_counts = np.bincount(train_targets)

# Effective Number of Samples (pour déséquilibre extrême)
beta = 0.9999
effective_num = 1.0 - np.power(beta, class_counts)
class_weights = (1.0 - beta) / effective_num
class_weights = class_weights / class_weights.sum() * len(class_counts)

print("📊 Class weights (Effective Number):")
for cls_idx, weight in enumerate(class_weights):
    print(f"   {train_dataset.idx_to_class[cls_idx]}: {weight:.2f}")

class_weights_tensor = torch.FloatTensor(class_weights)

# Weighted Random Sampler
sample_weights = [class_weights[target] for target in train_targets]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("\n✅ WeightedRandomSampler créé")

# ============================================================
# 6. DATALOADERS
# ============================================================

print("\n" + "="*70)
print("📦 CRÉATION DES DATALOADERS")
print("="*70)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    sampler=sampler,
    num_workers=CONFIG['num_workers'],
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"✅ Dataloaders créés:")
print(f"   Train: {len(train_loader)} batches")
print(f"   Val: {len(val_loader)} batches")

# Test d'un batch
test_imgs, test_labels = next(iter(train_loader))
print(f"\n🔍 Test batch:")
print(f"   Shape: {test_imgs.shape}")
print(f"   Range: [{test_imgs.min():.2f}, {test_imgs.max():.2f}]")

del test_imgs, test_labels
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ============================================================
# 7. FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):
    """Focal Loss pour déséquilibre extrême"""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(
            inputs, targets, reduction='none', weight=self.alpha
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss)
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

print("\n✅ Focal Loss définie")

# ============================================================
# 8. ARCHITECTURES DES MODÈLES
# ============================================================

print("\n" + "="*70)
print("🏗️ DÉFINITION DES ARCHITECTURES")
print("="*70)

num_classes = len(train_dataset.class_to_idx)

# ========== CNN PERSONNALISÉ ==========
class CustomCNN(nn.Module):
    """CNN optimisé pour images médicales"""
    def __init__(self, num_classes=4):
        super(CustomCNN, self).__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1),
            
            # Block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),
            
            # Block 3
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3),
            
            # Block 4
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.4),
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print("✅ CustomCNN défini")

# ========== RESNET-101 ==========
def get_resnet101(num_classes, pretrained=True):
    """ResNet-101 avec transfer learning"""
    model = models.resnet101(pretrained=pretrained)
    
    # Geler les premières couches
    for param in list(model.parameters())[:-30]:
        param.requires_grad = False
    
    # Remplacer le classifier
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.6),
        nn.Linear(num_ftrs, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes)
    )
    
    return model

print("✅ ResNet-101 défini")

# ========== VISION TRANSFORMER ==========
def get_vit(num_classes, pretrained=True):
    """Vision Transformer (ViT-B/16)"""
    try:
        model = models.vit_b_16(pretrained=pretrained)
        
        # Remplacer la tête de classification
        num_ftrs = model.heads.head.in_features
        model.heads.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(num_ftrs, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        
        return model
    except Exception as e:
        print(f"⚠️ Erreur ViT: {e}")
        return None

print("✅ ViT défini")

# ============================================================
# 9. FONCTION D'ENTRAÎNEMENT
# ============================================================

def train_model(model, train_loader, val_loader, criterion, optimizer, 
                scheduler, num_epochs, device, model_name):
    """Entraîne un modèle avec métriques détaillées"""
    
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_balanced_acc = 0.0
    
    history = {
        'train_loss': [], 'train_acc': [], 'train_balanced_acc': [],
        'val_loss': [], 'val_acc': [], 'val_balanced_acc': [],
        'learning_rates': []
    }
    
    print(f"\n{'='*70}")
    print(f"🚀 ENTRAÎNEMENT: {model_name}")
    print(f"{'='*70}")
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        print(f'\nÉpoque {epoch+1}/{num_epochs}')
        print('-' * 60)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader
            
            running_loss = 0.0
            all_preds = []
            all_labels = []
            
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            # Métriques
            epoch_loss = running_loss / len(dataloader.dataset)
            acc = accuracy_score(all_labels, all_preds)
            balanced_acc = balanced_accuracy_score(all_labels, all_preds)
            
            # Sauvegarder l'historique
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(acc)
                history['train_balanced_acc'].append(balanced_acc)
                history['learning_rates'].append(optimizer.param_groups[0]['lr'])
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(acc)
                history['val_balanced_acc'].append(balanced_acc)
            
            print(f'{phase:5s} | Loss: {epoch_loss:.4f} | Acc: {acc:.4f} | Bal.Acc: {balanced_acc:.4f}')
            
            # Sauvegarder le meilleur modèle
            if phase == 'val' and balanced_acc > best_balanced_acc:
                best_balanced_acc = balanced_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                print(f'      ✅ Nouveau meilleur! Bal.Acc: {best_balanced_acc:.4f}')
        
        # Scheduler
        if scheduler:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(history['val_balanced_acc'][-1])
            else:
                scheduler.step()
        
        epoch_time = time.time() - epoch_start
        print(f'Temps: {epoch_time//60:.0f}m {epoch_time%60:.0f}s')
        
        # Sauvegarder checkpoint
        if (epoch + 1) % 5 == 0:
            checkpoint_name = f'checkpoint_{model_name.lower().replace(" ", "_")}_epoch{epoch+1}.pth'
            torch.save(model.state_dict(), checkpoint_name)
            print(f'💾 Checkpoint: {checkpoint_name}')
    
    total_time = time.time() - since
    print(f'\n{"="*70}')
    print(f'✅ Entraînement terminé en {total_time//60:.0f}m {total_time%60:.0f}s')
    print(f'🏆 Meilleure Balanced Accuracy: {best_balanced_acc:.4f}')
    print(f'{"="*70}')
    
    # Charger les meilleurs poids
    model.load_state_dict(best_model_wts)
    
    return model, history

# ============================================================
# 10. FONCTION D'ÉVALUATION
# ============================================================

def evaluate_model(model, dataloader, device, class_names):
    """Évaluation complète avec toutes les métriques"""
    
    print("\n📊 Évaluation détaillée...")
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Métriques
    acc = accuracy_score(all_labels, all_preds)
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(
        all_labels, all_preds, target_names=class_names, zero_division=0
    )
    
    print(f"\n📈 Résultats:")
    print(f"   Accuracy: {acc:.4f}")
    print(f"   Balanced Accuracy: {balanced_acc:.4f} ⭐")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall: {recall:.4f}")
    print(f"   F1-Score: {f1:.4f}")
    
    print(f"\n{report}")
    
    return {
        'accuracy': acc,
        'balanced_accuracy': balanced_acc,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm,
        'classification_report': report,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs
    }

# ============================================================
# 11. ENTRAÎNEMENT DES 3 MODÈLES
# ============================================================

print("\n" + "="*70)
print("🎯 ENTRAÎNEMENT DES 3 MODÈLES")
print("="*70)

class_names = list(train_dataset.class_to_idx.keys())
models_results = {}
histories = {}

# ========== 1. CNN ==========
print("\n" + "🔵"*35)
print("MODÈLE 1/3: Custom CNN")
print("🔵"*35)

try:
    model_cnn = CustomCNN(num_classes=num_classes).to(device)
    criterion_cnn = FocalLoss(alpha=class_weights_tensor.to(device), gamma=2.0)
    optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=CONFIG['learning_rate_cnn'])
    scheduler_cnn = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_cnn, mode='max', factor=0.5, patience=3, verbose=True
    )
    
    model_cnn, history_cnn = train_model(
        model_cnn, train_loader, val_loader, criterion_cnn, optimizer_cnn,
        scheduler_cnn, CONFIG['num_epochs'], device, 'CNN'
    )
    
    # Sauvegarder
    torch.save(model_cnn.state_dict(), 'best_cnn.pth')
    print("\n💾 Modèle sauvegardé: best_cnn.pth")
    
    # Évaluer
    results_cnn = evaluate_model(model_cnn, val_loader, device, class_names)
    
    models_results['CNN'] = results_cnn
    histories['CNN'] = history_cnn
    
    # Libérer mémoire
    del model_cnn
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
except Exception as e:
    print(f"❌ Erreur CNN: {e}")
    import traceback
    traceback.print_exc()

# ========== 2. RESNET-101 ==========
print("\n" + "🟢"*35)
print("MODÈLE 2/3: ResNet-101")
print("🟢"*35)

try:
    model_resnet = get_resnet101(num_classes=num_classes, pretrained=True).to(device)
    criterion_resnet = FocalLoss(alpha=class_weights_tensor.to(device), gamma=2.0)
    optimizer_resnet = optim.Adam(model_resnet.parameters(), lr=CONFIG['learning_rate_resnet'])
    scheduler_resnet = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_resnet, mode='max', factor=0.5, patience=3, verbose=True
    )
    
    model_resnet, history_resnet = train_model(
        model_resnet, train_loader, val_loader, criterion_resnet, optimizer_resnet,
        scheduler_resnet, CONFIG['num_epochs'], device, 'ResNet-101'
    )
    
    # Sauvegarder
    torch.save(model_resnet.state_dict(), 'best_resnet101.pth')
    print("\n💾 Modèle sauvegardé: best_resnet101.pth")
    
    # Évaluer
    results_resnet = evaluate_model(model_resnet, val_loader, device, class_names)
    
    models_results['ResNet-101'] = results_resnet
    histories['ResNet-101'] = history_resnet
    
    # Libérer mémoire
    del model_resnet
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
except Exception as e:
    print(f"❌ Erreur ResNet: {e}")
    import traceback
    traceback.print_exc()

# ========== 3. VIT ==========
print("\n" + "🟣"*35)
print("MODÈLE 3/3: Vision Transformer")
print("🟣"*35)

try:
    model_vit = get_vit(num_classes=num_classes, pretrained=True)
    
    if model_vit is not None:
        model_vit = model_vit.to(device)
        criterion_vit = FocalLoss(alpha=class_weights_tensor.to(device), gamma=2.0)
        optimizer_vit = optim.AdamW(
            model_vit.parameters(), 
            lr=CONFIG['learning_rate_vit'], 
            weight_decay=0.01
        )
        scheduler_vit = optim.lr_scheduler.CosineAnnealingLR(
            optimizer_vit, T_max=CONFIG['num_epochs']
        )
        
        model_vit, history_vit = train_model(
            model_vit, train_loader, val_loader, criterion_vit, optimizer_vit,
            scheduler_vit, CONFIG['num_epochs'], device, 'ViT'
        )
        
        # Sauvegarder
        torch.save(model_vit.state_dict(), 'best_vit.pth')
        print("\n💾 Modèle sauvegardé: best_vit.pth")
        
        # Évaluer
        results_vit = evaluate_model(model_vit, val_loader, device, class_names)
        
        models_results['ViT'] = results_vit
        histories['ViT'] = history_vit
        
        # Libérer mémoire
        del model_vit
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("⚠️ ViT non disponible")
    
except Exception as e:
    print(f"❌ Erreur ViT: {e}")
    import traceback
    traceback.print_exc()

# ============================================================
# 12. COMPARAISONS ET VISUALISATIONS
# ============================================================

print("\n" + "="*70)
print("📊 COMPARAISONS DES MODÈLES")
print("="*70)

if len(models_results) == 0:
    print("❌ Aucun modèle entraîné avec succès")
else:
    # ========== Tableau comparatif ==========
    print("\n📋 TABLEAU COMPARATIF:")


🚀 ENTRAÎNEMENT - CNN, RESNET, VIT
   Utilisation des données prétraitées

🖥️  Configuration:
   Device: cpu

📋 Chargement de la configuration...
   ⚠️ Fichier de configuration introuvable

⚙️ Hyperparamètres:
   Batch size: 32
   Époques: 25
   Val split: 20%

🎨 DÉFINITION DES TRANSFORMATIONS
✅ Transformations définies
   Train: Augmentation + Normalize
   Val: Normalize seulement

✂️ CRÉATION DU SPLIT TRAIN/VALIDATION

📊 Dataset complet:
   Total: 11521 images
   Classes: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

   Distribution:
      MildDemented: 1665 (14.5%)
      ModerateDemented: 64 (0.6%)
      NonDemented: 5712 (49.6%)
      VeryMildDemented: 4080 (35.4%)

✅ Split stratifié:
   Train: 9216 images (80%)
   Val: 2305 images (20%)

⚖️ CALCUL DES CLASS WEIGHTS ET SAMPLER
📊 Class weights (Effective Number):
   MildDemented: 0.15
   ModerateDemented: 3.73
   NonDemented: 0.05
   VeryMildDemented: 0.07

✅ WeightedRandomSampler créé

📦 CRÉATION DES DATA

In [ ]:
# ============================================================
# ENTRAÎNEMENT AVEC RESNET ET COMPARAISON DE MODÈLES AVANCÉS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
from transformers import ViTForImageClassification, ViTImageProcessor
import timm  # pour les modèles avancés
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CONFIGURATION GÉNÉRALE
# ============================================================

CONFIG = {
    'num_epochs': 50,
    'batch_size': 32,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'num_classes': 4,
    'img_size': 224,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'model_save_path': 'MultimodalAI/Alzheimer/models/',
    'log_dir': 'MultimodalAI/Alzheimer/logs/'
}

# Créer les dossiers de sauvegarde
import os
os.makedirs(CONFIG['model_save_path'], exist_ok=True)
os.makedirs(CONFIG['log_dir'], exist_ok=True)

print(f"🚀 Configuration pour l'entraînement:")
print(f"   Device: {CONFIG['device']}")
print(f"   Epochs: {CONFIG['num_epochs']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Learning rate: {CONFIG['learning_rate']}")


In [ ]:

# ============================================================
# 2. CHARGEMENT DES DONNÉES AVEC TRANSFORMATIONS OPTIMALES
# ============================================================

# Transformations spécifiques pour chaque modèle
def get_transforms(model_type='resnet'):
    if model_type == 'vit':
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
    else:  # resnet, cnn, etc.
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

# Chargement des datasets (en utilisant vos images prétraitées)
PRETRAITE_TRAIN_DIR = "MultimodalAI/Alzheimer/data/Pretraite/train"
PRETRAITE_TEST_DIR = "MultimodalAI/Alzheimer/data/Pretraite/test"

train_dataset = AlzheimerDataset(PRETRAITE_TRAIN_DIR, transform=get_transforms('resnet'))
test_dataset = AlzheimerDataset(PRETRAITE_TEST_DIR, transform=get_transforms('resnet'))

# DataLoaders avec gestion du déséquilibre
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"📦 Données chargées:")
print(f"   Train: {len(train_dataset)} images")
print(f"   Test: {len(test_dataset)} images")
print(f"   Classes: {list(train_dataset.class_to_idx.keys())}")


In [ ]:

# ============================================================
# 3. MODÈLES AVANCÉS - IMPLÉMENTATION
# ============================================================

class AdvancedAlzheimerModels:
    def __init__(self, num_classes=4):
        self.num_classes = num_classes
        self.device = CONFIG['device']
    
    def get_resnet152(self):
        """ResNet-152 de Microsoft"""
        model = models.resnet152(pretrained=True)
        # Remplacer la dernière couche
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(model.fc.in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, self.num_classes)
        )
        return model.to(self.device)
    
    def get_resnet101(self):
        """ResNet-101 de Microsoft"""
        model = models.resnet101(pretrained=True)
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(model.fc.in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, self.num_classes)
        )
        return model.to(self.device)
    
    def get_vit_transformer(self):
        """Vision Transformer (ViT)"""
        try:
            # Version Hugging Face
            model = ViTForImageClassification.from_pretrained(
                'google/vit-base-patch16-224',
                num_labels=self.num_classes,
                ignore_mismatched_sizes=True
            )
            return model.to(self.device)
        except:
            # Fallback avec timm
            model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=self.num_classes)
            return model.to(self.device)
    
    def get_efficientnet(self):
        """EfficientNet-B4 - CNN moderne"""
        model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=self.num_classes)
        return model.to(self.device)
    
    def get_convnext(self):
        """ConvNeXt - Architecture moderne"""
        model = timm.create_model('convnext_base', pretrained=True, num_classes=self.num_classes)
        return model.to(self.device)
    
    def get_ensemble_model(self):
        """Modèle ensemble simple"""
        class EnsembleModel(nn.Module):
            def __init__(self, models_list):
                super().__init__()
                self.models = nn.ModuleList(models_list)
                self.classifier = nn.Linear(self.num_classes * len(models_list), self.num_classes)
            
            def forward(self, x):
                outputs = [model(x) for model in self.models]
                if isinstance(outputs[0], dict):  # Pour ViT HuggingFace
                    outputs = [output.logits for output in outputs]
                concatenated = torch.cat(outputs, dim=1)
                return self.classifier(concatenated)
        
        # Créer un ensemble avec ResNet101 et EfficientNet
        model1 = self.get_resnet101()
        model2 = self.get_efficientnet()
        ensemble = EnsembleModel([model1, model2])
        return ensemble.to(self.device)


In [ ]:

# ============================================================
# 4. ENTRAÎNEMENT ET ÉVALUATION
# ============================================================

class AlzheimerTrainer:
    def __init__(self, model, model_name):
        self.model = model
        self.model_name = model_name
        self.device = CONFIG['device']
        self.writer = SummaryWriter(f"{CONFIG['log_dir']}/{model_name}_{int(time.time())}")
        
        # Loss avec poids de classe
        self.criterion = nn.CrossEntropyLoss()
        
        # Optimizer
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr=CONFIG['learning_rate'],
            weight_decay=CONFIG['weight_decay']
        )
        
        # Scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=10, T_mult=2
        )
        
        self.history = {
            'train_loss': [], 'train_acc': [], 'train_f1': [],
            'test_loss': [], 'test_acc': [], 'test_f1': [],
            'learning_rates': []
        }
    
    def train_epoch(self, train_loader):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"Training {self.model_name}")
        for images, labels in pbar:
            images, labels = images.to(self.device), labels.to(self.device)
            
            self.optimizer.zero_grad()
            
            outputs = self.model(images)
            if isinstance(outputs, dict):  # Pour ViT HuggingFace
                outputs = outputs.logits
            
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}", 
                'acc': f"{100 * correct / total:.2f}%"
            })
        
        epoch_loss = running_loss / total
        epoch_acc = 100 * correct / total
        
        return epoch_loss, epoch_acc
    
    def evaluate(self, test_loader):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                
                outputs = self.model(images)
                if isinstance(outputs, dict):  # Pour ViT HuggingFace
                    outputs = outputs.logits
                
                loss = self.criterion(outputs, labels)
                running_loss += loss.item() * images.size(0)
                
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        epoch_loss = running_loss / total
        epoch_acc = 100 * correct / total
        
        # Calcul F1-score
        from sklearn.metrics import f1_score
        f1 = f1_score(all_labels, all_preds, average='weighted')
        
        return epoch_loss, epoch_acc, f1, all_labels, all_preds
    
    def train(self, train_loader, test_loader, num_epochs):
        print(f"\n🚀 Début de l'entraînement pour {self.model_name}")
        print("=" * 60)
        
        best_acc = 0.0
        
        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print("-" * 50)
            
            # Entraînement
            train_loss, train_acc = self.train_epoch(train_loader)
            
            # Évaluation
            test_loss, test_acc, test_f1, _, _ = self.evaluate(test_loader)
            
            # Scheduler
            self.scheduler.step()
            
            # Sauvegarde historique
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['test_loss'].append(test_loss)
            self.history['test_acc'].append(test_acc)
            self.history['test_f1'].append(test_f1)
            self.history['learning_rates'].append(self.optimizer.param_groups[0]['lr'])
            
            # TensorBoard
            self.writer.add_scalar('Loss/train', train_loss, epoch)
            self.writer.add_scalar('Loss/test', test_loss, epoch)
            self.writer.add_scalar('Accuracy/train', train_acc, epoch)
            self.writer.add_scalar('Accuracy/test', test_acc, epoch)
            self.writer.add_scalar('F1/test', test_f1, epoch)
            self.writer.add_scalar('Learning_rate', self.optimizer.param_groups[0]['lr'], epoch)
            
            # Affichage
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
            print(f"Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.2f}%")
            print(f"Test F1:    {test_f1:.4f}")
            print(f"LR:         {self.optimizer.param_groups[0]['lr']:.2e}")
            
            # Sauvegarde meilleur modèle
            if test_acc > best_acc:
                best_acc = test_acc
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'best_acc': best_acc,
                    'history': self.history
                }, f"{CONFIG['model_save_path']}/best_{self.model_name}.pth")
                print(f"✅ Meilleur modèle sauvegardé! (Acc: {best_acc:.2f}%)")
        
        self.writer.close()
        return self.history


In [ ]:

# ============================================================
# 5. COMPARAISON DE TOUS LES MODÈLES
# ============================================================

def compare_all_models():
    """Compare tous les modèles avancés sur le dataset Alzheimer"""
    
    print("\n" + "="*70)
    print("🔬 COMPARAISON DES MODÈLES AVANCÉS POUR LA DÉTECTION ALZHEIMER")
    print("="*70)
    
    model_factory = AdvancedAlzheimerModels(num_classes=CONFIG['num_classes'])
    
    # Liste des modèles à tester
    models_to_test = {
        'ResNet-152': model_factory.get_resnet152,
        'ResNet-101': model_factory.get_resnet101,
        'Vision-Transformer': model_factory.get_vit_transformer,
        'EfficientNet-B4': model_factory.get_efficientnet,
        'ConvNeXt': model_factory.get_convnext,
        # 'Ensemble': model_factory.get_ensemble_model  # Optionnel
    }
    
    results = {}
    training_histories = {}
    
    for model_name, model_func in models_to_test.items():
        print(f"\n🎯 Entraînement du modèle: {model_name}")
        print("-" * 50)
        
        try:
            # Charger le modèle
            model = model_func()
            
            # Compter les paramètres
            total_params = sum(p.numel() for p in model.parameters())
            trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"   Paramètres totaux: {total_params:,}")
            print(f"   Paramètres entraînables: {trainable_params:,}")
            
            # Entraînement
            trainer = AlzheimerTrainer(model, model_name)
            history = trainer.train(train_loader, test_loader, CONFIG['num_epochs'])
            
            # Résultats finaux
            final_test_acc = history['test_acc'][-1]
            final_test_f1 = history['test_f1'][-1]
            best_test_acc = max(history['test_acc'])
            
            results[model_name] = {
                'final_accuracy': final_test_acc,
                'best_accuracy': best_test_acc,
                'final_f1': final_test_f1,
                'parameters': total_params
            }
            
            training_histories[model_name] = history
            
            print(f"\n✅ {model_name} terminé:")
            print(f"   Best Accuracy: {best_test_acc:.2f}%")
            print(f"   Final F1: {final_test_f1:.4f}")
            
        except Exception as e:
            print(f"❌ Erreur avec {model_name}: {e}")
            continue
    
    return results, training_histories


In [ ]:

# ============================================================
# 6. VISUALISATION DES RÉSULTATS
# ============================================================

def visualize_comparison(results, training_histories):
    """Visualise la comparaison des modèles"""
    
    print("\n" + "="*70)
    print("📊 VISUALISATION DES RÉSULTATS")
    print("="*70)
    
    # Graphique de comparaison des performances
    fig, axes = plt.subplots(2, 2, figsize=(20, 15))
    fig.suptitle('COMPARAISON DES MODÈLES - DÉTECTION ALZHEIMER', 
                 fontsize=16, fontweight='bold')
    
    # 1. Accuracy finale
    model_names = list(results.keys())
    final_accuracies = [results[name]['final_accuracy'] for name in model_names]
    best_accuracies = [results[name]['best_accuracy'] for name in model_names]
    
    x = np.arange(len(model_names))
    width = 0.35
    
    bars1 = axes[0, 0].bar(x - width/2, final_accuracies, width, label='Final', alpha=0.7)
    bars2 = axes[0, 0].bar(x + width/2, best_accuracies, width, label='Best', alpha=0.7)
    
    axes[0, 0].set_xlabel('Modèles')
    axes[0, 0].set_ylabel('Accuracy (%)')
    axes[0, 0].set_title('Accuracy Finale vs Meilleure', fontweight='bold')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(model_names, rotation=45)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    for bar in bars1:
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.1f}%', ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.1f}%', ha='center', va='bottom', fontsize=8)
    
    # 2. F1-score vs Paramètres
    f1_scores = [results[name]['final_f1'] for name in model_names]
    parameters = [results[name]['parameters'] for name in model_names]
    
    scatter = axes[0, 1].scatter(parameters, f1_scores, s=100, alpha=0.7)
    axes[0, 1].set_xlabel('Nombre de Paramètres')
    axes[0, 1].set_ylabel('F1-Score')
    axes[0, 1].set_title('F1-Score vs Complexité du Modèle', fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Ajouter les labels
    for i, name in enumerate(model_names):
        axes[0, 1].annotate(name, (parameters[i], f1_scores[i]), 
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # 3. Courbes d'apprentissage (Accuracy)
    for model_name, history in training_histories.items():
        axes[1, 0].plot(history['test_acc'], label=model_name, linewidth=2)
    
    axes[1, 0].set_xlabel('Epochs')
    axes[1, 0].set_ylabel('Accuracy (%)')
    axes[1, 0].set_title('Évolution de l\'Accuracy (Test)', fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Courbes de loss
    for model_name, history in training_histories.items():
        axes[1, 1].plot(history['test_loss'], label=model_name, linewidth=2)
    
    axes[1, 1].set_xlabel('Epochs')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].set_title('Évolution de la Loss (Test)', fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('comparaison_modeles_alzheimer.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Rapport détaillé
    print("\n📋 RAPPORT DÉTAILLÉ DES PERFORMANCES:")
    print("-" * 60)
    for model_name, result in results.items():
        print(f"\n🏆 {model_name}:")
        print(f"   📈 Best Accuracy: {result['best_accuracy']:.2f}%")
        print(f"   🎯 Final Accuracy: {result['final_accuracy']:.2f}%")
        print(f"   ⚖️  F1-Score: {result['final_f1']:.4f}")
        print(f"   🔧 Paramètres: {result['parameters']:,}")


In [ ]:

# ============================================================
# 7. LANCEMENT DE LA COMPARAISON
# ============================================================

if __name__ == "__main__":
    print("🧠 DÉTECTION ALZHEIMER - COMPARAISON DE MODÈLES AVANCÉS")
    print("=" * 70)
    
    # Lancer la comparaison
    results, training_histories = compare_all_models()
    
    # Visualiser les résultats
    visualize_comparison(results, training_histories)
    
    # Conclusion
    print("\n" + "="*70)
    print("🎯 CONCLUSIONS ET RECOMMANDATIONS")
    print("="*70)
    
    if results:
        best_model = max(results.items(), key=lambda x: x[1]['best_accuracy'])
        print(f"\n🏆 MEILLEUR MODÈLE: {best_model[0]}")
        print(f"   Accuracy: {best_model[1]['best_accuracy']:.2f}%")
        print(f"   F1-Score: {best_model[1]['final_f1']:.4f}")
        
        print(f"\n💡 RECOMMANDATIONS:")
        print(f"   1. Utiliser {best_model[0]} pour la production")
        print(f"   2. Vision Transformer excellent pour les images médicales")
        print(f"   3. ResNet-152 bon équilibre performance/complexité")
        print(f"   4. EfficientNet efficace avec moins de paramètres")
        
        print(f"\n📊 PERFORMANCE MOYENNE SUR TOUS LES MODÈLES:")
        avg_accuracy = np.mean([r['best_accuracy'] for r in results.values()])
        avg_f1 = np.mean([r['final_f1'] for r in results.values()])
        print(f"   📈 Accuracy moyenne: {avg_accuracy:.2f}%")
        print(f"   ⚖️  F1-Score moyen: {avg_f1:.4f}")
    
    print("\n" + "="*70)
    print("✨ COMPARAISON TERMINÉE - PRÊT POUR LE DÉPLOIEMENT!")
    print("="*70)